<a href="https://colab.research.google.com/github/Minh-Quan-hub/AAI2025/blob/Fall_2026_MinhQuan/Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code generation with prompt chaining using ChatGPT

**The prompt use:**

Create Python code for a customer-support bot using the Gemini API key.

The bot should use the following prompt chain:

Prompt 1: Read the customer’s message and identify the type of problem they are experiencing.

Prompt 2: Based on the identified problem, determine what additional information is needed from the customer.

Prompt 3: Using the customer’s problem and additional information, provide a helpful solution or transfer the case to a human support agent.

Make sure the output from each prompt is used as input for the next prompt. Allow the user to enter a customer message and display the result from each step.

In [4]:
from google import genai
from google.colab import userdata

# Load the Gemini API key from Google Colab Secrets
try:
    api_key = userdata.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)
except Exception:
    raise ValueError(
        "Gemini API key not found. Add GEMINI_API_KEY to Colab Secrets."
    )

MODEL_NAME = "gemini-3.6-flash"


# Send a prompt to Gemini
def ask_gemini(prompt):
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )
        return response.text
    except Exception as error:
        return f"Gemini API error: {error}"


# Prompt 1: Identify the customer's problem
def identify_problem(customer_message):
    prompt = f"""
You are a customer-support assistant.

Read the customer's message and identify the main problem.
Classify it as delivery, damaged product, wrong item, refund,
payment, account, or other.

Customer message:
{customer_message}

Return the problem type and a short explanation.
"""
    return ask_gemini(prompt)


# Prompt 2: Determine what information is needed
def find_missing_information(customer_message, problem_output):
    prompt = f"""
You are continuing a customer-support case.

Customer message:
{customer_message}

Problem identified in Prompt 1:
{problem_output}

Determine what additional information is needed to help the customer.
Ask no more than two short questions.
Do not request passwords or full payment-card information.
"""
    return ask_gemini(prompt)


# Prompt 3: Create the final customer-support response
def create_final_response(customer_message, problem_output, info_output):
    prompt = f"""
Create a helpful response for this customer.

Original customer message:
{customer_message}

Problem identified in Prompt 1:
{problem_output}

Information needed from Prompt 2:
{info_output}

Write a polite customer-support response under 100 words.
Explain the next step clearly. If more information is needed, ask
the questions from Prompt 2. If the issue involves fraud or a payment
dispute, transfer the case to a human support agent. Do not claim that you
checked an order, contacted a carrier, issued a refund, or completed another
action unless that action actually occurred. If information is missing, explain
that a support agent can investigate after the customer provides it.
"""
    return ask_gemini(prompt)


# Get a message from the user
customer_message = input("Enter the customer's message: ")

# Run the prompt chain
step_1 = identify_problem(customer_message)
step_2 = find_missing_information(customer_message, step_1)
step_3 = create_final_response(customer_message, step_1, step_2)

# Display each step
print("\n--- Prompt 1: Identified Problem ---")
print(step_1)

print("\n--- Prompt 2: Additional Information Needed ---")
print(step_2)

print("\n--- Prompt 3: Final Customer-Support Response ---")
print(step_3)

Enter the customer's message: I ordered a laptop charger five days ago, but my tracking information has not changed and the package still has not arrived.

--- Prompt 1: Identified Problem ---
**Problem Type:** Delivery

**Explanation:** The customer has not received their package, and the tracking information has remained unchanged for five days after placing the order.

--- Prompt 2: Additional Information Needed ---
I'm sorry to hear that your charger hasn't arrived yet. To help me look into this for you, could you please provide:

1. What is your order number?
2. What is the email address associated with your order (or the tracking number)?

--- Prompt 3: Final Customer-Support Response ---
I am sorry to hear that your laptop charger has not arrived yet. 

Once you provide a few details, a support agent can investigate the shipment status for you. Could you please share:

1. What is your order number?
2. What is the email address associated with your order (or the tracking number)?